# VascuQuest JAX Virtual Disease Backend — One-Subject Qualification

This notebook qualifies the optional JAX execution backend introduced in PR #20 against the frozen NumPy Virtual Disease solver.

Exactly **one canonical PWDB subject** is used across all four disease conditions. For every condition the NumPy and JAX RHS/stability operators are compared on the same deterministic non-trivial state, followed by a complete JAX 116-segment solve. One large-artery-stiffening case is additionally solved end-to-end with NumPy and compared across the full network.

A PASS is software/mechanistic backend qualification only. Outputs remain `MODELLED`; this is not clinical validation or an epidemiological study.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = 'https://github.com/KNOWDYN/VascuQuest.git'
QUALIFICATION_REF = 'release/parameterized-cohort-qualification'
LOCAL_REPO = Path('/content/VascuQuest-jax-qualification')
LOCAL_SOURCE = Path('/content/vascuquest-pwdb-source')
LOCAL_XDG = Path('/content/vascuquest-xdg')
OUTPUT_BASE = Path('/content/drive/MyDrive/VascuQuest/jax_one_subject_qualification')

drive_candidates = [
    Path('/content/drive/MyDrive/VQ_WallWork_CBM/source/PWDB_3275625'),
    Path('/content/drive/Shareddrives/VQ_WallWork_CBM/source/PWDB_3275625'),
]
DRIVE_SOURCE = next((p for p in drive_candidates if p.exists()), drive_candidates[0])

for path in (LOCAL_SOURCE, LOCAL_XDG, OUTPUT_BASE):
    path.mkdir(parents=True, exist_ok=True)
os.environ['XDG_DATA_HOME'] = str(LOCAL_XDG / 'data')
os.environ['XDG_CACHE_HOME'] = str(LOCAL_XDG / 'cache')
os.environ['XDG_STATE_HOME'] = str(LOCAL_XDG / 'state')

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)
subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', QUALIFICATION_REF, REPO_URL, str(LOCAL_REPO)],
    check=True,
)
CODE_REVISION = subprocess.check_output(
    ['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True
).strip()
OUTPUT_ROOT = OUTPUT_BASE / CODE_REVISION[:12]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(LOCAL_REPO) + '[jax]'],
    check=True,
)
import jax

print('Code revision:', CODE_REVISION)
print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())
print('Configured PWDB Drive source:', DRIVE_SOURCE)
print('Revision-scoped output:', OUTPUT_ROOT)


## Stage the three canonical PWDB artifacts to local SSD

There is no recursive Drive search. Exact files in `PWDB_3275625` are copied once; a missing artifact such as `geo.zip` is acquired canonically by VascuQuest.

In [ ]:
STAGER = LOCAL_REPO / 'tests/full_data/parameterized_cohort_colab_stage.py'
STAGE_REPORT = OUTPUT_ROOT / 'source_stage.json'
stage = subprocess.run([
    sys.executable, str(STAGER),
    '--drive-source-dir', str(DRIVE_SOURCE),
    '--local-source', str(LOCAL_SOURCE),
    '--report', str(STAGE_REPORT),
])
if stage.returncode != 0:
    raise RuntimeError(f'PWDB staging failed with exit code {stage.returncode}')
print(STAGE_REPORT.read_text())
print('PWDB local-SSD source gate: PASS')


## Run the one-subject JAX qualification

The same subject is used for carotid stenosis, iliac stenosis, fusiform AAA and large-artery stiffening. Progress is printed condition-by-condition.

In [ ]:
RUNNER = LOCAL_REPO / 'tests/full_data/jax_one_subject_qualification.py'
REPORT = OUTPUT_ROOT / 'jax-one-subject-qualification.json'
completed = subprocess.run([
    sys.executable, str(RUNNER),
    '--source', str(LOCAL_SOURCE),
    '--report', str(REPORT),
    '--code-revision', CODE_REVISION,
])
if completed.returncode != 0:
    if REPORT.exists():
        print('Persisted failure record:')
        print(REPORT.read_text())
    raise RuntimeError(f'JAX qualification failed with exit code {completed.returncode}')


## Qualification summary

In [ ]:
report = json.loads(REPORT.read_text())
summary = {
    'status': report.get('status'),
    'code_revision': report.get('code_revision'),
    'canonical_subject_id': report.get('canonical_subject_id'),
    'source_age_years': report.get('source_age_years'),
    'elapsed_seconds': report.get('elapsed_seconds'),
    'anchor': report.get('full_numpy_jax_anchor'),
}
print(json.dumps(summary, indent=2, sort_keys=True))
for case in report.get('cases', []):
    timing = case['jax_full_solve'].get('timing', {})
    print(
        case['condition'],
        'operator=', case['operator_equivalence']['passed'],
        'converged=', case['jax_full_solve']['diagnostics']['converged'],
        'wall_s=', round(case['jax_full_solve']['wall_seconds'], 3),
        'device=', timing.get('device'),
    )
if report.get('status') != 'PASS':
    raise RuntimeError('PR #20 JAX qualification is not PASS')
print('PR #20 JAX one-subject qualification: PASS')
print('Durable report:', REPORT)


After the final cell prints `PASS`, the durable JSON report under the revision-scoped Drive directory is the evidence to inspect before merging PR #20. A PASS qualifies the JAX backend implementation against the frozen NumPy mechanics for this one-subject gate; it does not establish clinical validity.